# Notebook 02: Huấn luyện và Diễn giải Cây Quyết định (Decision Tree)

**Người phụ trách chính:** TV2 (Decision Tree & Logistic Regression).

### Mục tiêu nghiên cứu:
1. Xây dựng pipeline hoàn chỉnh gồm tiền xử lý (One-Hot, Imputation) và bộ phân loại `DecisionTreeClassifier`.
2. Khám phá không gian siêu tham số: `criterion` (gini, entropy), `max_depth` (3, 5, 8, None), `min_samples_leaf` (1, 5, 15), và `class_weight` (None, balanced) qua 5-Fold Stratified CV trên tập phát triển.
3. Trực quan hóa cấu trúc cây phân nhánh, rút trích luật quyết định từ gốc đến lá và phân tích trade-off giữa độ sâu cây và nguy cơ quá khớp (overfitting).


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập import thư viện nội bộ
sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))
from telco_churn.data import load_raw_data
from telco_churn.schema import validate_records, ALL_FEATURE_COLUMNS, TARGET_COLUMN
from telco_churn.training import run_grid_search, build_model_pipeline
from telco_churn.evaluation import plot_decision_tree_structure, compute_classification_metrics

sns.set_theme(style='whitegrid')
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']


## 1. Nạp tập phát triển (Development Set: 80% — N=5.634)
Tuân thủ nghiêm ngặt giao thức Plan 03: Chỉ sử dụng dữ liệu phát triển từ `split_manifest.csv`.


In [ ]:
df_raw = load_raw_data('../telco-customer-churn-on-icp4d/data/Telco-Customer-Churn.csv')
valid_df, _ = validate_records(df_raw, mode='training')
manifest = pd.read_csv('../data/splits/split_manifest.csv')

dev_manifest = manifest[manifest['split'] == 'dev']
dev_df = valid_df.merge(dev_manifest[['customerID', 'fold']], on='customerID')
X_dev = dev_df[ALL_FEATURE_COLUMNS].copy()
y_dev = dev_df[TARGET_COLUMN].copy()
folds = dev_df['fold'].values
print(f'Số mẫu tập phát triển: {len(dev_df):,} | Số đặc trưng đầu vào: {X_dev.shape[1]}')


## 2. Tìm kiếm siêu tham số bằng 5-Fold Stratified CV
Tìm kiếm qua 48 cấu hình theo Plan 03 với tiêu chí refit là F1-Score lớp 1.


In [ ]:
dt_res = run_grid_search('decision_tree', X_dev, y_dev, folds)
print('Cấu hình tối ưu (Best Params):', dt_res['best_params'])
print(f'Mean CV F1 lớp 1: {dt_res["mean_f1"]:.4f} ± {dt_res["std_f1"]:.4f}')
print(f'Mean CV Recall: {dt_res["mean_recall"]:.4f} | ROC-AUC: {dt_res["mean_roc_auc"]:.4f}')


## 3. Trực quan hóa Cấu trúc Cây Quyết định (Độ sâu hiển thị: 3 tầng)
Quan sát các biến phân nhánh hàng đầu tại các tầng cao nhất của cây.


In [ ]:
fig = plot_decision_tree_structure(dt_res['fitted_pipeline'], max_depth=3)
plt.show()


## 4. Rút trích Luật quyết định và Nhận định Nghiệp vụ
- **Nút gốc (Root):** Thuộc tính phân chia quan trọng nhất là loại hợp đồng `Contract = Month-to-month` kết hợp thời gian gắn bó `tenure`.
- **Luật Churn điển hình:** Nếu khách hàng có hợp đồng tháng (`Month-to-month`), sử dụng Internet cáp quang (`InternetService = Fiber optic`) và thời gian gắn bó dưới 6 tháng thì tỷ lệ rời bỏ tăng lên trên 65%.
- **Ưu điểm:** Khả năng giải thích luật trắng (white-box) rõ ràng, phù hợp trình bày với bộ phận kinh doanh.
- **Hạn chế:** Cây đơn lẻ có phương sai cao; nếu không giới hạn `max_depth` cây sẽ mở rộng quá sâu dẫn đến overfitting nghiêm trọng.
